In [1]:
# -*- coding: utf-8 -*-
"""Laboratory Work No. 5: Probabilistic Primality Testing Algorithms"""

import random

In [2]:
def mod_pow(base, exponent, mod):
    """
    Fast modular exponentiation
    Computes (base^exponent) % mod efficiently
    """
    result = 1
    base = base % mod

    while exponent > 0:
        # If exponent is odd, multiply result by base
        if exponent % 2 == 1:
            result = (result * base) % mod

        # Square the base and halve the exponent
        base = (base * base) % mod
        exponent = exponent // 2

    return result

In [3]:
def fermat_test(n, k=5):
    """
    Fermat primality test
    n: number to test
    k: number of test rounds
    Returns: True if n is probably prime, False if composite
    """
    # Handle small numbers
    if n < 2:
        return False
    if n in [2, 3]:
        return True
    if n % 2 == 0:
        return False

    # Perform k rounds of testing
    for _ in range(k):
        # Choose random a in [2, n-2]
        a = random.randint(2, n - 2)

        # Fermat's little theorem: a^(n-1) ≡ 1 (mod n) for primes
        if mod_pow(a, n - 1, n) != 1:
            return False

    # Passed all tests
    return True

In [11]:
def jacobi_symbol(a, n):
    """
    Compute the Jacobi symbol (a/n)
    a: integer
    n: positive odd integer
    Returns: -1, 0, or 1
    """
    # Step 1: Initialization
    if a == 0:
        return 0
    if a == 1:
        return 1

    result = 1
    a = a % n

    # Step 2: Remove factors of 2 from a
    while a % 2 == 0:
        a = a // 2
        # Check quadratic reciprocity for factor 2
        if n % 8 == 3 or n % 8 == 5:
            result = -result

    # Step 3: Apply quadratic reciprocity
    if a == 1:
        return result

    # If both are 3 mod 4, flip sign
    if a % 4 == 3 and n % 4 == 3:
        result = -result

    # Step 4: Swap a and n and continue
    return result * jacobi_symbol(n % a, a)

In [12]:
def solovay_strassen_test(n, k=5):
    """
    Solovay-Strassen primality test
    n: number to test
    k: number of test rounds
    Returns: True if n is probably prime, False if composite
    """
    # Handle small numbers
    if n < 2:
        return False
    if n in [2, 3]:
        return True
    if n % 2 == 0:
        return False

    # Perform k rounds of testing
    for _ in range(k):
        # Choose random a in [2, n-1]
        a = random.randint(2, n - 1)

        # Compute Jacobi symbol
        jacobi = jacobi_symbol(a, n)

        # If Jacobi symbol is 0, n is composite
        if jacobi == 0:
            return False

        # Compute a^((n-1)/2) mod n
        exponent = (n - 1) // 2
        mod_val = mod_pow(a, exponent, n)

        # Adjust mod_val to be in range [-1, 0, 1] for comparison
        if mod_val == n - 1:
            mod_val = -1
        elif mod_val > 1:
            mod_val = 1

        # Check Euler criterion: a^((n-1)/2) ≡ (a/n) (mod n)
        if mod_val != jacobi:
            return False

    # Passed all tests
    return True

In [13]:
def miller_rabin_test(n, k=5):
    """
    Miller-Rabin primality test
    n: number to test
    k: number of test rounds
    Returns: True if n is probably prime, False if composite
    """
    # Handle small numbers
    if n < 2:
        return False
    if n in [2, 3]:
        return True
    if n % 2 == 0:
        return False

    # Write n-1 as d*2^s with d odd
    s = 0
    d = n - 1
    while d % 2 == 0:
        d = d // 2
        s += 1

    # Perform k rounds of testing
    for _ in range(k):
        # Choose random a in [2, n-2]
        a = random.randint(2, n - 2)

        # Compute x = a^d mod n
        x = mod_pow(a, d, n)

        # If x == 1 or x == n-1, this round passes
        if x == 1 or x == n - 1:
            continue

        # Otherwise, square x repeatedly
        composite = True
        for _ in range(s - 1):
            x = mod_pow(x, 2, n)

            # If we hit n-1, we can stop this round
            if x == n - 1:
                composite = False
                break

            # If we hit 1, n is composite
            if x == 1:
                return False

        # If we never hit n-1, n is composite
        if composite:
            return False

    # Passed all tests
    return True

In [14]:
def test_all_algorithms():
    """Test all primality testing algorithms with examples"""

    print("=" * 70)
    print("LABORATORY WORK No. 5")
    print("Testing Probabilistic Primality Algorithms")
    print("=" * 70)

    # Test cases: (number, is_prime)
    test_cases = [
        (2, True),      # prime
        (3, True),      # prime
        (4, False),     # composite
        (5, True),      # prime
        (7, True),      # prime
        (9, False),     # composite
        (11, True),     # prime
        (13, True),     # prime
        (15, False),    # composite
        (17, True),     # prime
        (19, True),     # prime
        (21, False),    # composite
        (23, True),     # prime
        (29, True),     # prime
        (561, False),   # Carmichael number (composite)
        (1105, False),  # Carmichael number (composite)
        (1729, False),  # Carmichael number (composite)
        (2465, False),  # Carmichael number (composite)
        (2821, False),  # Carmichael number (composite)
        (6601, False),  # Carmichael number (composite)
        (8911, False),  # Carmichael number (composite)
        (999983, True), # prime
    ]

    print(f"\n{'Number':<10} {'Expected':<10} {'Fermat':<10} {'Solovay-Strassen':<18} {'Miller-Rabin':<12} {'Result':<10}")
    print("-" * 80)

    for n, expected in test_cases:
        # Test each algorithm
        fermat_result = fermat_test(n, k=3)
        solovay_result = solovay_strassen_test(n, k=3)
        miller_result = miller_rabin_test(n, k=3)

        # All algorithms should agree
        all_match = (fermat_result == solovay_result == miller_result)
        correct = (fermat_result == expected)

        # Format results
        fermat_str = "Prime" if fermat_result else "Composite"
        solovay_str = "Prime" if solovay_result else "Composite"
        miller_str = "Prime" if miller_result else "Composite"
        expected_str = "Prime" if expected else "Composite"
        result_str = "✓" if correct else "✗"

        print(f"{n:<10} {expected_str:<10} {fermat_str:<10} {solovay_str:<18} {miller_str:<12} {result_str:<10}")

    print("=" * 70)
    print("Note: Fermat test may fail on Carmichael numbers")
    print("Solovay-Strassen and Miller-Rabin are more reliable")
    print("=" * 70)

In [15]:
def main():
    """Main program function"""

    print("LABORATORY WORK No. 5")
    print("Probabilistic Primality Testing Algorithms")
    print()
    print("Algorithms implemented:")
    print("  - Fermat test")
    print("  - Solovay-Strassen test")
    print("  - Miller-Rabin test")
    print()

    # Run tests
    test_all_algorithms()

    print("\n" + "=" * 70)
    print("Program finished")

if __name__ == "__main__":
    main()

LABORATORY WORK No. 5
Probabilistic Primality Testing Algorithms

Algorithms implemented:
  - Fermat test
  - Solovay-Strassen test
  - Miller-Rabin test

LABORATORY WORK No. 5
Testing Probabilistic Primality Algorithms

Number     Expected   Fermat     Solovay-Strassen   Miller-Rabin Result    
--------------------------------------------------------------------------------
2          Prime      Prime      Prime              Prime        ✓         
3          Prime      Prime      Prime              Prime        ✓         
4          Composite  Composite  Composite          Composite    ✓         
5          Prime      Prime      Prime              Prime        ✓         
7          Prime      Prime      Prime              Prime        ✓         
9          Composite  Composite  Composite          Composite    ✓         
11         Prime      Prime      Prime              Prime        ✓         
13         Prime      Prime      Prime              Prime        ✓         
15         Com